In [ ]:
import pandas as pd
train_identity  = pd.read_csv('ieee-fraud-detection/train_identity.csv')
train_transaction  = pd.read_csv('ieee-fraud-detection/train_transaction.csv')
test_identity = pd.read_csv('ieee-fraud-detection/test_identity.csv')
test_transaction = pd.read_csv('ieee-fraud-detection/test_transaction.csv')
# Normalize column names (replace '-' with '_')
train_identity.columns = train_identity.columns.str.replace('-', '_')
test_identity.columns = test_identity.columns.str.replace('-', '_')


In [ ]:
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', None)        
pd.set_option('display.expand_frame_repr', False)

In [ ]:
# Merge transaction and identity data on TransactionID
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')
display(train.head())
display(test.head())

In [ ]:
print(train.shape, test.shape)
print(train['card1'].isnull().sum())



# graphs
let's look at some interesting observations

In [ ]:
import matplotlib.pyplot as plt

def plot_user2_fraud_analysis(df):
    df = df.copy()
    df['card2_isnull'] = df['card2'].isnull()
    
    # Count frauds and non-frauds per user2 null status
    fraud_stats = df.groupby(['card2_isnull', 'isFraud']).size().unstack(fill_value=0)
    
    # Fraud count and fraud rate
    fraud_counts = fraud_stats[1]
    total_counts = fraud_stats.sum(axis=1)
    fraud_rate = fraud_counts / total_counts

    # Plotting
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Absolute fraud count
    fraud_counts.plot(kind='bar', ax=ax[0], color='crimson')
    ax[0].set_title('Fraud Count by card2 Null Status')
    ax[0].set_xticklabels(['Not Null', 'Null'], rotation=0)
    ax[0].set_ylabel('Fraud Count')

    # 2. Fraud rate
    fraud_rate.plot(kind='bar', ax=ax[1], color='darkorange')
    ax[1].set_title('Fraud Rate by card2 Null Status')
    ax[1].set_xticklabels(['Not Null', 'Null'], rotation=0)
    ax[1].set_ylabel('Fraud Rate')

    # 3. Ratio of fraud vs non-fraud
    fraud_stats_ratio = fraud_stats.div(fraud_stats.sum(axis=1), axis=0)
    fraud_stats_ratio.plot(kind='bar', stacked=True, ax=ax[2], color=['skyblue', 'firebrick'])
    ax[2].set_title('Fraud vs Not Fraud Ratio by card2 Null Status')
    ax[2].set_xticklabels(['Not Null', 'Null'], rotation=0)
    ax[2].set_ylabel('Proportion')
    ax[2].legend(['Not Fraud', 'Fraud'])

    plt.tight_layout()
    plt.show()

plot_user2_fraud_analysis(train)  
#we see that there is a clear difference when user2 is null

# feature engineering

In [ ]:
import pandas as pd

# Fake start date (since we don't know the real one)
start_date = pd.to_datetime('2017-01-01')

def add_transaction_datetime_features(df):
    df = df.copy()
    df['TransactionDate'] = start_date + pd.to_timedelta(df['TransactionDT'], unit='s')
    
    df['Trans_weekday'] = df['TransactionDate'].dt.weekday  # Monday=0
    df['Trans_hour'] = df['TransactionDate'].dt.hour
    df['Trans_day'] = df['TransactionDate'].dt.day
    df['Trans_month'] = df['TransactionDate'].dt.month
    df['Trans_weekofyear'] = df['TransactionDate'].dt.isocalendar().week

    df = df.drop(columns=['TransactionDate'])  # Remove if you don't want the full datetime
    return df


def add_transaction_amount_features(df, user_id_col):
    df = df.copy()
    
    # Only legitimate transactions
    legit = df
    
    # Compute mean TransactionAmt per user
    user_avg_amt = legit.groupby(user_id_col)['TransactionAmt'].mean().rename('User_Avg_TransactionAmt')
    
    # Merge it back
    df = df.merge(user_avg_amt, left_on=user_id_col, right_index=True, how='left')
    
    # (Optional) Ratio of current amount to user's average
    df['Amt_to_avg_ratio'] = df['TransactionAmt'] / df['User_Avg_TransactionAmt']
    
    return df

def add_more_transaction_features(df):
    df = df.copy()
    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
    df['TransactionAmt_mod_1'] = df['TransactionAmt'] % 1  # Cents
    df['Small_Transaction'] = (df['TransactionAmt'] < 10).astype(int)
    return df


In [ ]:
import numpy as np

print(train.shape, test.shape)
train = add_transaction_datetime_features(train)
test = add_transaction_datetime_features(test)

train = add_transaction_amount_features(train, user_id_col='card1')
test = add_transaction_amount_features(test, user_id_col='card1')

train = add_more_transaction_features(train)
test = add_more_transaction_features(test)
print(train.shape, test.shape)


# graphs

In [ ]:

# Dynamically identify categorical features based on data types
categorical_features = train.select_dtypes(include=['object']).columns.tolist()

# Identify numeric features
numeric_features = train.select_dtypes(include=[np.number]).columns.tolist()
# Exclude 'TransactionID' and 'isFraud' from numeric features
numeric_features = [col for col in numeric_features if col not in ['TransactionID', 'isFraud']]
print(f"numeric features: \n{numeric_features}")

In [ ]:
# Function to compute statistics
def compute_statistics(df, name):
    stats = pd.DataFrame(index=df.columns)
    stats['missing_values'] = df.isnull().sum()
    stats['missing_percentage'] = 100 * stats['missing_values'] / len(df)
    stats['mean'] = df[numeric_features].mean()
    stats['unique_categories'] = df[categorical_features].nunique()
    stats['dataset'] = name
    return stats

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Compute statistics for train and test datasets
train_stats = compute_statistics(train, 'train')
test_stats = compute_statistics(test, 'test')

# Combine statistics
combined_stats = pd.concat([train_stats, test_stats])
# display(combined_stats)
# Reset index for plotting
combined_stats = combined_stats.reset_index().rename(columns={'index': 'feature'})

# Plotting missing percentage per feature
plt.figure(figsize=(15, 10))
sns.barplot(data=combined_stats, x='missing_percentage', y='feature', hue='dataset')
plt.title('Missing Percentage per Feature')
plt.xlabel('Missing Percentage')
plt.ylabel('Feature')
plt.legend(title='Dataset')
plt.tight_layout()
plt.show()

In [ ]:

# Plot mean of numeric features
mean_stats = combined_stats.dropna(subset=['mean'])
plt.figure(figsize=(15, 10))
sns.barplot(data=mean_stats, x='mean', y='feature', hue='dataset')
plt.title('Mean of Numeric Features')
plt.xlabel('Mean')
plt.ylabel('Feature')
plt.legend(title='Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Plot unique categories of categorical features
unique_cat_stats = combined_stats.dropna(subset=['unique_categories'])
plt.figure(figsize=(15, 10))
sns.barplot(data=unique_cat_stats, x='unique_categories', y='feature', hue='dataset')
plt.title('Unique Categories per Categorical Feature')
plt.xlabel('Number of Unique Categories (log scale)')
plt.ylabel('Feature')
plt.xscale('log')  # <-- log scale on x-axis
plt.legend(title='Dataset')
plt.tight_layout()
plt.show()


# useful methods

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
import pandas as pd

class NAFiller:
    def __init__(self, num_strategy='constant', num_fill_value=9999,
                 cat_strategy='constant', cat_fill_value='Missing'):
        self.num_strategy = num_strategy
        self.num_fill_value = num_fill_value
        self.cat_strategy = cat_strategy
        self.cat_fill_value = cat_fill_value
        self.num_imputer = None
        self.cat_imputer = None
        self.num_cols = []
        self.cat_cols = []

    def fit(self, X, y=None):
        # Identify numerical and categorical columns
        self.num_cols = X.select_dtypes(include=['number']).columns.tolist()
        self.cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        
        # Fit numerical imputer
        self.num_imputer = SimpleImputer(strategy=self.num_strategy, fill_value=self.num_fill_value)
        if self.num_cols:
            self.num_imputer.fit(X[self.num_cols])
        
        # Fit categorical imputer
        self.cat_imputer = SimpleImputer(strategy=self.cat_strategy, fill_value=self.cat_fill_value)
        if self.cat_cols:
            self.cat_imputer.fit(X[self.cat_cols])

        return self

    def transform(self, X):
        X = X.copy()
        # Transform numerical columns
        if self.num_cols:
            X[self.num_cols] = self.num_imputer.transform(X[self.num_cols])
        # Transform categorical columns
        if self.cat_cols:
            X[self.cat_cols] = self.cat_imputer.transform(X[self.cat_cols])
        return X


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class Encoder(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=3):
        self.threshold = threshold
        self.ohe_cols = []
        self.woe_cols = []
        self.woe_maps = {}
        self.ohe_df_columns = []

    def fit(self, X, y):
        X = X.copy()
        y = pd.Series(y).reset_index(drop=True)
        
        for col in X.select_dtypes(include=['object', 'category']):
            unique_vals = X[col].nunique(dropna=True)
            if unique_vals < self.threshold:
                self.ohe_cols.append(col)
            else:
                self.woe_cols.append(col)
                self.woe_maps[col] = self._compute_woe(X[col], y)

        if self.ohe_cols:
            ohe_df = pd.get_dummies(X[self.ohe_cols], dummy_na=True)
            self.ohe_df_columns = ohe_df.columns.tolist()
        
        return self

    def transform(self, X):
        X = X.copy()
        output = X.drop(columns=self.ohe_cols + self.woe_cols, errors='ignore')

        # Apply WOE encoding
        for col in self.woe_cols:
            woe_map = self.woe_maps[col]
            X[col] = X[col].map(woe_map).fillna(0)  # fill unknown categories with 0
            output[col] = X[col]

        # Apply One-Hot Encoding
        if self.ohe_cols:
            ohe_df = pd.get_dummies(X[self.ohe_cols], dummy_na=True)
            # Align with training columns to ensure consistency
            ohe_df = ohe_df.reindex(columns=self.ohe_df_columns, fill_value=0)
            output = pd.concat([output, ohe_df], axis=1)

        return output

    def _compute_woe(self, feature_col, target_col):
        df = pd.DataFrame({'feature': feature_col, 'target': target_col})
        df = df.dropna(subset=['feature'])  # Drop missing for WOE mapping
        grouped = df.groupby('feature')

        # total good and bad
        total_good = (df['target'] == 0).sum()
        total_bad = (df['target'] == 1).sum()

        woe_map = {}
        for val, group in grouped:
            good = (group['target'] == 0).sum()
            bad = (group['target'] == 1).sum()

            # Avoid division by zero
            good_ratio = good / total_good if total_good else 1e-6
            bad_ratio = bad / total_bad if total_bad else 1e-6
            woe = 0
            if bad_ratio > 0:
                woe = np.log(good_ratio / bad_ratio) 
            woe_map[val] = woe

        return woe_map


In [ ]:
# Split train
train_ids = train["TransactionID"]
train_y = train["isFraud"]
train_X = train.drop(columns=["TransactionID", "isFraud"])

# Split test
test_ids = test["TransactionID"]
test_X = test.drop(columns=["TransactionID"], errors="ignore")  # in case 'isFraud' is not present


# fill NA values

In [ ]:
filler = NAFiller(
    num_strategy='constant', 
    num_fill_value=999, #I think that when I use 9999 and more it overflows and causes errors so i chose 999
    cat_strategy='constant', 
    cat_fill_value='Missing'
)

filler.fit(X=train_X)
train_filled = filler.transform(train_X)
test_filled = filler.transform(test_X)
train_filled.head()
print(train_filled.isna().sum().sum())

# encoding

In [ ]:

encoder = Encoder(threshold=6)

# Fit only on train
encoder.fit(X=train_filled, y=train_y)
train_encoded = encoder.transform(X=train_filled)
test_encoded = encoder.transform(test_filled)

In [ ]:
print(train_filled.shape)
print(train_encoded.shape)
train_encoded.head()

# correlation filter

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle

class CorrelationFilter:
    def __init__(self, threshold=0.9, matrix_path='correlation_matrix.csv', columns_path='columns_to_keep.pkl'):
        """
        Initializes the correlation filter with optional save paths.

        Parameters:
        - threshold: float, correlation threshold to drop one of two correlated features
        - matrix_path: str, file path to save the correlation matrix
        - columns_path: str, file path to save the list of columns to keep
        """
        self.threshold = threshold
        self.matrix_path = matrix_path
        self.columns_path = columns_path
        self.corr_matrix_ = None
        self.columns_to_keep_ = None
    def fit(self, X: pd.DataFrame, y: pd.Series):
        if os.path.exists(self.matrix_path) and os.path.exists(self.columns_path):
            self.corr_matrix_ = pd.read_csv(self.matrix_path, index_col=0)
            with open(self.columns_path, 'rb') as f:
                self.columns_to_keep_ = pickle.load(f)
            return self
    
        # Drop all-NaN and constant columns
        X = X.loc[:, X.notna().any()]  # Remove all-NaN columns
        X = X.loc[:, X.nunique() > 1]  # Remove constant columns
    
        corr_matrix = X.corr().abs()
        self.corr_matrix_ = corr_matrix
        corr_matrix.to_csv(self.matrix_path)
    
        target_corr = X.apply(lambda col: np.corrcoef(col, y)[0, 1])
    
        columns_to_drop = set()
        for i in range(len(corr_matrix.columns)):
            for j in range(i + 1, len(corr_matrix.columns)):
                col1, col2 = corr_matrix.columns[i], corr_matrix.columns[j]
                if corr_matrix.loc[col1, col2] > self.threshold:
                    if abs(target_corr[col1]) >= abs(target_corr[col2]):
                        columns_to_drop.add(col2)
                    else:
                        columns_to_drop.add(col1)
    
        self.columns_to_keep_ = [col for col in X.columns if col not in columns_to_drop]
    
        with open(self.columns_path, 'wb') as f:
            pickle.dump(self.columns_to_keep_, f)
    
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.columns_to_keep_ is None:
            raise ValueError("Must call fit() before transform()")
        return X[self.columns_to_keep_].copy()

    def fit_transform(self, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
        self.fit(X, y)
        return self.transform(X)


# data processing

In [ ]:
# drop useless columns
print(train_encoded.shape)
single_value_cols = train_encoded.columns[train_encoded.nunique() <= 1].tolist()

train_encoded = train_encoded.drop(columns=single_value_cols)
test_encoded = test_encoded.drop(columns=single_value_cols)  # ignore if not present
print(train_encoded.shape)
print(test_encoded.shape)


In [ ]:
train_encoded.head()

In [ ]:

filterer = CorrelationFilter(threshold=0.89)
filterer.fit(X=train_encoded, y=train_y)
train_filtered = filterer.transform(X=train_encoded)
test_filtered = filterer.transform(test_encoded)

# Access saved correlation matrix and columns
cor_matrix = filterer.corr_matrix_
columns_used = filterer.columns_to_keep_


In [ ]:
print(train_encoded.nunique())